In [1]:
import fsspec
import xarray as xr
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import helpers

# Todos for this notebook

**NOTE:** This work depends on https://github.com/zarr-developers/VirtualiZarr/pull/369

- [x] DRY up some functions, perhaps separate to another file
- [x] Extract tests for kerchunk and hdfreader and encoding to appendices
- [x] Switch glob to file by date function
- [x] Come up with a function for validation
- [ ] Execute as S3 storage
- [ ] Estimate time to complete 2004-2024

Nice to have:
- [ ] Add arraylake option

# 1. Start a dask cluster

The dask cluster will help parallelize generating references and in computation for validation.

In [2]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster()
client = Client(cluster)
cluster.scale(8)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/abarciauskas-bgse/proxy/8787/status,
Dashboard: /user/abarciauskas-bgse/proxy/8787/status,Workers: 4
Total threads: 16,Total memory: 60.62 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43509,Workers: 4
Dashboard: /user/abarciauskas-bgse/proxy/8787/status,Total threads: 16
Started: Just now,Total memory: 60.62 GiB
Comm: tcp://127.0.0.1:39833,Total threads: 4
Dashboard: /user/abarciauskas-bgse/proxy/35829/status,Memory: 15.16 GiB
Nanny: tcp://127.0.0.1:33881,


# 2. Initialize file stores for reading and writing

## 2a. Initialize a filesystem for accessing the MUR SST data files.

In [3]:
fs = fsspec.filesystem("s3", anon=False)

## 2b. Initialize the store we are writing to (icechunk).

**NOTE:** If just appending to the store, `overwrite` should `=False`.

If overwriting an existing s3 store, you need to run the following lines:

<code>
!pip install awscli
!aws s3 rm --recursive s3://nasa-veda-scratch/icechunk/{store_name}
</code>

In [4]:
store = helpers.find_or_create_icechunk_store(store_name="MUR-JPL-L4-GLOB-v4.1-virtual", store_type="s3", overwrite=False)
store

# 3. Create initial store with data from 2002

## 3a. List, virtualize and concatenize datasets

This step uses the dmrpp reader of VirtualiZarr. This reader makes this process very fast since we don't actually have to open and read any of the original files.

In [ ]:
mur_sst_files_2002 = helpers.list_mur_sst_files(start_date="2002-06-01", end_date="2002-12-31")
mur_sst_dmrpps_2002 = [f + '.dmrpp' for f in mur_sst_files_2002]
virtual_ds_2002 = helpers.create_virtual_ds(dmrpps=mur_sst_dmrpps_2002)

In [ ]:
# sanity check
len(mur_sst_dmrpps_2002)

## 3b. Write to icechunk

In [ ]:
%%time
virtual_ds_2002.virtualize.to_icechunk(store)
store.commit("Wrote 2002 data")

## 3c. Validate

In [ ]:
helpers.validate_data(store, dates=["2002-06-01", "2002-12-31"], fs=fs)

# 4. Append 2003

One file in 2003 (2003-09-11) had a different encoding, so the the list of 2003 files is split into 3 lists. See 2a for how problematic date was found. All dates apart from the date with the different encoding are written as virtual stores. The problematic data is written as zarr.

See and run `helpers.get_codecs` with a list of virtual datasets to check all codecs are the same.

## 4a. List files from 2003

And split that list by the date with the different encoding.

In [ ]:
mur_sst_files_2003_1 = helpers.list_mur_sst_files(start_date="2003-01-01", end_date="2003-09-10")

## 4b. Write first set of files as virtual datasets using the DMRPP reader

In [ ]:
mur_sst_files_2003_1_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2003_1]
virtual_ds_2003_1 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2003_1_dmrpps)

In [ ]:
virtual_ds_2003_1.virtualize.to_icechunk(store, append_dim='time')
store.commit("Wrote first part of 2003 data")

## 4c. Write data with different encoding as zarr

In [ ]:
# this takes about a minute and a lot of memory (nearly 40GB)
problematic_file = helpers.list_mur_sst_files(start_date="2003-09-11", end_date="2003-09-11")
# using chunks={} or chunks='auto' to initialize the dataset with dask arrays fails, throwing an error that the store is in read-only mode.
# I have not investigated this.
ds = xr.open_dataset(fs.open(problematic_file[0]))
ds.to_zarr(store, append_dim='time')
store.commit(f"Wrote {problematic_file} in zarr")

## 4d. Write the rest of 2003 as virtual data

In [ ]:
mur_sst_files_2003_2 = helpers.list_mur_sst_files(start_date="2003-09-12", end_date="2003-12-31")
mur_sst_files_2003_2_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2003_2]
virtual_ds_2003_2 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2003_2_dmrpps)

In [ ]:
virtual_ds_2003_2.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote to end of 2003.")

## 4e. Validate

In [ ]:
helpers.validate_data(store, dates=["2003-01-01", "2003-12-31"], fs=fs)

# 5. Append 2004

## 5a. List files

In [ ]:
dates = ['2004-01-01', '2004-12-31']
mur_sst_files_2004 = helpers.list_mur_sst_files(start_date=dates[0], end_date=dates[1])
mur_sst_files_2004_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2004]

In [ ]:
len(mur_sst_files_2004_dmrpps)

## 5b. Write data

In [ ]:
virtual_ds_2004 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2004_dmrpps)
virtual_ds_2004.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote 2004 to store.")

## 5c. Validate data

In [ ]:
%%time
helpers.validate_data(store, dates=dates, fs=fs)

# 6. Let's try 2 years! 2005-2006

## 6a. List files

In [ ]:
dates = ['2005-01-01', '2006-12-31']
mur_sst_files_2005_2006 = helpers.list_mur_sst_files(start_date=dates[0], end_date=dates[1])
mur_sst_files_2005_2006_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2005_2006]

In [ ]:
len(mur_sst_files_2005_2006_dmrpps)

## 6b. Write data

In [ ]:
virtual_ds_2005_2006 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2005_2006_dmrpps)
virtual_ds_2005_2006.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote 2005-2006 to store.")

## 6c. Validate data

In [ ]:
%%time
helpers.validate_data(store, dates=dates, fs=fs)

# 7. Let's try 5 years! 2007 through end of 2011

## 7a. List files

In [ ]:
dates = ['2007-01-01', '2011-12-31']
mur_sst_files_2007_2011 = helpers.list_mur_sst_files(start_date=dates[0], end_date=dates[1])
mur_sst_files_2007_2011_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2007_2011]

In [ ]:
len(mur_sst_files_2007_2011_dmrpps)

## 7b. Write data

In [ ]:
virtual_ds_2007_2011 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2007_2011_dmrpps)
virtual_ds_2007_2011.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote 2007-2011 to store.")

## 7c. Validate data

In [ ]:
%%time
helpers.validate_data(store, dates=dates, fs=fs)

# 8. 2012

## 8a. List files

In [67]:
dates = ['2012-01-01', '2012-12-31']
mur_sst_files_2012 = helpers.list_mur_sst_files(start_date=dates[0], end_date=dates[1])
mur_sst_files_2012_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2012]

In [68]:
len(mur_sst_files_2012_dmrpps)

366

## 8b. Write data

In [69]:
virtual_ds_2012 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2012_dmrpps)

In [72]:
virtual_ds_2012.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote 2012 to store.")

'SJCBH5V3RH4TK4RMXX60'

## 8c. Validate data

In [73]:
%%time
helpers.validate_data(store, dates=dates, fs=fs)

Open icechunk store...
Computing icechunk store result...
Icechunk store result: 283.4256386034788
Opening original files...
Computing original files result
Result from original files: 283.4256386034788
CPU times: user 1min 47s, sys: 5.01 s, total: 1min 52s
Wall time: 6min 18s


# 9. 2013

## 9a. List files

In [75]:
dates = ['2013-01-01', '2013-12-31']
mur_sst_files_2013 = helpers.list_mur_sst_files(start_date=dates[0], end_date=dates[1])
mur_sst_files_2013_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2013]

In [76]:
len(mur_sst_files_2013_dmrpps)

365

## 9b. Write data

In [77]:
virtual_ds_2013 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2013_dmrpps)

In [78]:
virtual_ds_2013.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote 2013 to store.")

'T002PMA3NC5CNHFAKT20'

## 9c. Validate data

In [6]:
%%time
dates = ['2013-01-01', '2013-12-31']
helpers.validate_data(store, dates=dates, fs=fs)

Open icechunk store...
Computing icechunk store result...
Icechunk store result: 282.54163894462664
Opening original files...
Computing original files result
Result from original files: 282.54163894462664
CPU times: user 42.5 s, sys: 5.26 s, total: 47.7 s
Wall time: 3min 50s


# 10. 2014

## 10a. List files

In [8]:
dates = ['2014-01-01', '2014-12-31']
mur_sst_files_2014 = helpers.list_mur_sst_files(start_date=dates[0], end_date=dates[1])
mur_sst_files_2014_dmrpps = [f + '.dmrpp' for f in mur_sst_files_2014]

In [9]:
len(mur_sst_files_2014_dmrpps)

365

In [16]:
# encodings = {}
# first_ds = xr.open_dataset(fs.open(mur_sst_files_2014[0]))
# for var in first_ds.data_vars:
#     encoding = first_ds[var].encoding.copy()
#     encoding.pop('source', None)
#     encodings[var] = encoding

# for f in mur_sst_files_2014:
#     print(f"Opening file {f}")
#     ds = xr.open_dataset(fs.open(f))
#     for var in ds.data_vars:
#         current_encoding = ds[var].encoding
#         current_encoding.pop('source', None)
#         if current_encoding != encodings[var]:
#             import pdb; pdb.set_trace()
#     print("")

In [21]:
xr.open_dataset(fs.open(mur_sst_files_2014[1]))

<xarray.Dataset> Size: 29GB
Dimensions:           (time: 1, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 8B 2014-01-02T09:00:00
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    analysed_sst      (time, lat, lon) float64 5GB ...
    analysis_error    (time, lat, lon) float64 5GB ...
    mask              (time, lat, lon) float32 3GB ...
    sea_ice_fraction  (time, lat, lon) float64 5GB ...
    dt_1km_data       (time, lat, lon) timedelta64[ns] 5GB ...
    sst_anomaly       (time, lat, lon) float64 5GB ...
Attributes: (12/47)
    Conventions:                CF-1.7
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

In [20]:
xr.open_dataset(fs.open(mur_sst_files_2014[-2]))

<xarray.Dataset> Size: 18GB
Dimensions:           (time: 1, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 8B 2014-12-30T09:00:00
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    analysed_sst      (time, lat, lon) float64 5GB ...
    analysis_error    (time, lat, lon) float64 5GB ...
    mask              (time, lat, lon) float32 3GB ...
    sea_ice_fraction  (time, lat, lon) float64 5GB ...
Attributes: (12/47)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

## 10b. Write data

In [10]:
virtual_ds_2014 = helpers.create_virtual_ds(dmrpps=mur_sst_files_2014_dmrpps)

2025-01-06 00:03:46,782 - distributed.worker - ERROR - Compute Failed
Key:       ('from_sequence-open_virtual-reduce_via_concat-part-5c55e0920b25e16dd469b9e909b60bcc', 4)
State:     executing
Task:  <Task ('from_sequence-open_virtual-reduce_via_concat-part-5c55e0920b25e16dd469b9e909b60bcc', 4) _execute_subgraph(...)>
Exception: 'NotImplementedError("Doesn\'t support slicing with (array([ 0, -1]), slice(None, None, None), slice(None, None, None))")'
Traceback: '  File "/opt/conda/lib/python3.11/site-packages/dask/bag/core.py", line 2510, in empty_safe_apply\n    return func(part)\n           ^^^^^^^^^^\n  File "/home/jovyan/icechunk-nasa/helpers.py", line 19, in reduce_via_concat\n    return xr.concat(\n           ^^^^^^^^^^\n  File "/opt/conda/lib/python3.11/site-packages/xarray/core/concat.py", line 277, in concat\n    return _dataset_concat(\n           ^^^^^^^^^^^^^^^^\n  File "/opt/conda/lib/python3.11/site-packages/xarray/core/concat.py", line 674, in _dataset_concat\n    combined

NotImplementedError: Doesn't support slicing with (array([ 0, -1]), slice(None, None, None), slice(None, None, None))

In [ ]:
virtual_ds_2014.virtualize.to_icechunk(store, append_dim='time')
store.commit(f"Wrote 2014 to store.")

## 10c. Validate data

In [ ]:
%%time
helpers.validate_data(store, dates=dates, fs=fs)

# Post-script: Checking the store and how long it takes to open it.

In [5]:
%%time
xr.open_zarr(store, consolidated=False)

CPU times: user 14.7 s, sys: 4.79 s, total: 19.5 s
Wall time: 1min


<xarray.Dataset> Size: 77TB
Dimensions:           (time: 4234, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 34kB 2002-06-01T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    analysis_error    (time, lat, lon) float64 22TB dask.array<chunksize=(1, 1023, 2047), meta=np.ndarray>
    analysed_sst      (time, lat, lon) float64 22TB dask.array<chunksize=(1, 1023, 2047), meta=np.ndarray>
    sea_ice_fraction  (time, lat, lon) float64 22TB dask.array<chunksize=(1, 1447, 2895), meta=np.ndarray>
    mask              (time, lat, lon) float32 11TB dask.array<chunksize=(1, 1447, 2895), meta=np.ndarray>
Attributes: (12/47)
    Conventions:                CF-1.5
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    MUR = "Multi-scale Ultra-high Reolution"
    creator_email:              ghrsst@podaac.jpl.nasa.gov
    ...                         ...
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    time_coverage_end:          20130101T210000Z
    time_coverage_start:        20121231T210000Z
    title:                      Daily MUR SST, Final product
    uuid:                       27665bc0-d5fc-11e1-9b23-0800200c9a66
    westernmost_longitude:      -180.0